# EEG_07e — Pre-build Grafi ed Ipergrafi come Tensori

Costruisce e salva dataset PyG (.pt) per tutti i trial EEG.  
**Esegui dall'alto in basso.** Le ultime celle salvano i file su disco.

| File output | Contenuto |
|---|---|
| `graphs/dataset_pcc_k{K}.pt`  | grafi PCC k-NN, per-trial edge_index |
| `graphs/dataset_plv_k{K}.pt`  | grafi PLV k-NN, per-trial edge_index |
| `graphs/dataset_wpli_k{K}.pt` | grafi wPLI k-NN, per-trial edge_index |
| `graphs/dataset_hgnn_k{K}.pt` | ipergrafi PCC, per-trial hyperedge_index |

**Pruning disponibile:**
- **Edge pruning**: rimuovi archi dove connettività < soglia
- **Channel pruning**: rimuovi canali EEG con media PCC < mean - N·sigma
- Analisi visiva nella cella 4 prima di decidere


In [ ]:
# ════════════════════════════════════════════════════════════
#  CONFIGURAZIONE — modifica qui prima di eseguire
# ════════════════════════════════════════════════════════════

# Valori di k da costruire (edge_index k-NN)
K_VALUES       = [6]          # es. [4, 6, 8] per multi-k

# Metodi grafi semplici
METHODS        = ["pcc", "plv", "wpli"]

# Ipergrafi
BUILD_HGNN     = True         # costruisci anche dataset_hgnn_k{K}.pt

# Pruning archi — rimuovi archi dove connettività < EDGE_THRESHOLD
# 0.0 = nessun pruning (tutti gli archi k-NN inclusi)
EDGE_THRESHOLD = 0.0

# Pruning canali — rimuovi canali con media PCC < mean - N_SIGMA * std
# None = nessun pruning canali
# Imposta CHANNEL_DROP_NAMES se vuoi rimuovere canali specifici per nome
CHANNEL_PRUNING_SIGMA = None  # es. 2.0 per rimuovere canali molto deboli
CHANNEL_DROP_NAMES    = []    # es. ["A1", "A2"] se vuoi forzare rimozione

# Bande per PLV/wPLI
PLV_BANDS = [(4, 8), (8, 13)]  # theta + alpha

# Forza ricostruzione anche se file già esiste
FORCE_REBUILD = True

# ════════════════════════════════════════════════════════════
print("Config OK")
print(f"  K_VALUES:       {K_VALUES}")
print(f"  METHODS:        {METHODS}")
print(f"  BUILD_HGNN:     {BUILD_HGNN}")
print(f"  EDGE_THRESHOLD: {EDGE_THRESHOLD}")
print(f"  CHANNEL_SIGMA:  {CHANNEL_PRUNING_SIGMA}")
print(f"  CHANNEL_DROP:   {CHANNEL_DROP_NAMES}")


In [ ]:
import gc
import sys, time
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert as sp_hilbert
from torch_geometric.data import Data
from tqdm.auto import tqdm

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import get_keep_channels

META_CSV   = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH  = project_root / "src" / "io" / "ebneuro.locs"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# Canali: tutti e 61 dall'H5 (63 .locs - 2 non registrati Pz/POz)
keep_idx_base, keep_names_base = get_keep_channels(ELOC_PATH)
print(f"Canali base: {len(keep_idx_base)} — {keep_names_base[:5]}...")

meta = pd.read_csv(META_CSV)
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)].copy()
meta = meta[pd.to_numeric(meta["subject_id"], errors="coerce").notna()].copy()
meta["subject_id"] = meta["subject_id"].astype(int).astype(str).str.zfill(2)
print(f"Trial totali: {len(meta)}")

In [ ]:
# ════════════════════════════════════════════════════════════
# FUNZIONI CONNETTIVITÀ
# ════════════════════════════════════════════════════════════

def pcc_matrix(x_np):
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return pcc.astype(np.float32)

def plv_matrix(x_np, sfreq=256, bands=PLV_BANDS):
    nyq = sfreq / 2.0
    N = x_np.shape[0]
    plv_sum = np.zeros((N, N), dtype=np.float64)
    for flo, fhi in bands:
        b, a    = butter(4, [flo/nyq, fhi/nyq], btype="band")
        x_f     = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        phi     = np.angle(sp_hilbert(x_f, axis=1))
        exp_phi = np.exp(1j * phi)
        plv_sum += np.abs(exp_phi @ exp_phi.conj().T) / x_np.shape[1]
    plv = (plv_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(plv, 0.0)
    return plv

def wpli_matrix(x_np, sfreq=256, bands=PLV_BANDS):
    """wPLI vectorizzato con broadcasting numpy — evita il loop O(N²) Python."""
    nyq = sfreq / 2.0
    N = x_np.shape[0]
    wpli_sum = np.zeros((N, N), dtype=np.float64)
    for flo, fhi in bands:
        b, a     = butter(4, [flo/nyq, fhi/nyq], btype="band")
        x_f      = filtfilt(b, a, x_np, axis=1)
        analytic = sp_hilbert(x_f, axis=1)                                       # (N, T) complex
        # im_cs[i,j,t] = Im(analytic[i,t] * conj(analytic[j,t]))
        im_cs    = np.imag(analytic[:, None, :] * analytic.conj()[None, :, :])   # (N, N, T) float64
        # wPLI = |E[Im]| / E[|Im|]
        wpli_sum += np.abs(im_cs.mean(axis=-1)) / (np.abs(im_cs).mean(axis=-1) + 1e-8)
        del im_cs
    wpli = (wpli_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(wpli, 0.0)
    return wpli

def knn_edge_index(matrix, k, threshold=0.0):
    """k-NN con soglia opzionale. Ritorna LongTensor [2, E]."""
    N = matrix.shape[0]
    edges = set()
    for i in range(N):
        row = matrix[i].copy()
        row[row < threshold] = 0.0
        top = np.argsort(row)[::-1][:k]
        for j in top:
            if row[j] > 0.0:
                edges.add((i, int(j)))
                edges.add((int(j), i))
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long)
    s, d = zip(*sorted(edges))
    return torch.tensor([list(s), list(d)], dtype=torch.long)

def hyperedge_index(x_np, k, threshold=0.0):
    """Per-trial hyperedge_index da PCC. Ritorna [2, N*(k+1)]."""
    pcc = pcc_matrix(x_np)
    N   = pcc.shape[0]
    verts, edges = [], []
    for i in range(N):
        row = pcc[i].copy()
        row[row < threshold] = 0.0
        top = np.argsort(row)[::-1][:k]
        members = [i] + [int(j) for j in top if row[j] > 0.0]
        for v in members:
            verts.append(v); edges.append(i)
    return torch.tensor([verts, edges], dtype=torch.long)

print("Funzioni connettività OK")

In [ ]:
# ════════════════════════════════════════════════════════════
# ANALISI CANALI — media PCC per canale su campione di trial
# Usa questa cella per decidere il pruning PRIMA di costruire
# ════════════════════════════════════════════════════════════

N_SAMPLE_ANALYSIS = 300   # trial da campionare
rng = np.random.RandomState(42)
sample = meta.sample(min(N_SAMPLE_ANALYSIS, len(meta)), random_state=rng)

paths_map = defaultdict(list)
for _, row in sample.iterrows():
    paths_map[row["path_h5"]].append(int(row["epoch_idx"]))

pcc_sum   = np.zeros((len(keep_idx_base), len(keep_idx_base)), dtype=np.float64)
pcc_count = 0
for path, epoch_idxs in tqdm(paths_map.items(), desc="Analisi PCC", leave=False):
    with h5py.File(path, "r") as f:
        for e_idx in epoch_idxs:
            x_np = f["data"][e_idx][keep_idx_base, :].astype(np.float32)
            pcc_sum += pcc_matrix(x_np)
            pcc_count += 1

pcc_mean_global = (pcc_sum / pcc_count).astype(np.float32)
chan_connectivity = pcc_mean_global.mean(axis=1)  # connettività media per canale

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Heatmap PCC media ─────────────────────────────────────
sns.heatmap(pcc_mean_global, ax=axes[0], cmap="RdYlBu_r",
            vmin=0, vmax=0.6, xticklabels=False, yticklabels=False)
axes[0].set_title(f"PCC medio globale ({pcc_count} trial)\n[{len(keep_idx_base)} canali × {len(keep_idx_base)} canali]")

# ── Connettività per canale ───────────────────────────────
mu, sigma = chan_connectivity.mean(), chan_connectivity.std()
colors = ["#E74C3C" if c < mu - 1.5*sigma else "#1E2761" for c in chan_connectivity]
axes[1].bar(range(len(keep_idx_base)), chan_connectivity, color=colors, width=0.8)
axes[1].axhline(mu,           color="gray",    lw=1.5, ls="--", label=f"media={mu:.3f}")
axes[1].axhline(mu-1.5*sigma, color="#E74C3C", lw=1.5, ls=":",  label=f"mean-1.5σ={mu-1.5*sigma:.3f}")
axes[1].set_xticks(range(len(keep_idx_base)))
axes[1].set_xticklabels(keep_names_base, rotation=90, fontsize=6)
axes[1].set_title("Connettività media per canale (|PCC| medio verso tutti gli altri)")
axes[1].legend()

plt.tight_layout()
plt.savefig(project_root/"figures"/"eeg07e_channel_connectivity.png", dpi=150, bbox_inches="tight")
plt.show()

# Report canali deboli
print("\nCanali con connettività bassa (< mean - 1.5σ):")
weak_candidates = [(keep_names_base[i], chan_connectivity[i])
                   for i in range(len(keep_idx_base))
                   if chan_connectivity[i] < mu - 1.5*sigma]
for name, val in weak_candidates:
    print(f"  {name:6s}  conn={val:.4f}")
if not weak_candidates:
    print("  Nessuno — tutti i canali hanno connettività nella norma.")


In [ ]:
# ════════════════════════════════════════════════════════════
# APPLICA PRUNING — modifica CHANNEL_DROP_NAMES o
# CHANNEL_PRUNING_SIGMA nella cella 1, poi ri-esegui
# ════════════════════════════════════════════════════════════

keep_idx   = keep_idx_base.copy()
keep_names = keep_names_base.copy()

# Channel pruning per sigma
if CHANNEL_PRUNING_SIGMA is not None:
    mu, sigma = chan_connectivity.mean(), chan_connectivity.std()
    threshold_conn = mu - CHANNEL_PRUNING_SIGMA * sigma
    drop_by_sigma  = [i for i, c in enumerate(chan_connectivity) if c < threshold_conn]
    drop_names_sigma = [keep_names_base[i] for i in drop_by_sigma]
    print(f"Pruning sigma={CHANNEL_PRUNING_SIGMA}: rimuovo {drop_names_sigma}")
    keep_idx   = [keep_idx_base[i]   for i in range(len(keep_idx_base))   if i not in drop_by_sigma]
    keep_names = [keep_names_base[i] for i in range(len(keep_names_base)) if i not in drop_by_sigma]

# Channel pruning per nome esplicito
if CHANNEL_DROP_NAMES:
    drop_set   = set(CHANNEL_DROP_NAMES)
    keep_idx   = [keep_idx[i]   for i, n in enumerate(keep_names) if n not in drop_set]
    keep_names = [n             for n in keep_names                if n not in drop_set]
    print(f"Pruning esplicito: rimossi {CHANNEL_DROP_NAMES}")

N_CHANS = len(keep_idx)
print(f"\nCanali finali: {N_CHANS}  (erano {len(keep_idx_base)})")
print(f"Canali: {keep_names}")
print(f"Edge threshold: {EDGE_THRESHOLD}")


In [ ]:
# ════════════════════════════════════════════════════════════
# BUILD GRAFI SEMPLICI — PCC / PLV / wPLI
# x salvato grezzo (non normalizzato) — norm applicata in EEG_08
# y = label_idx grezzo (0-109) — mapping cluster in EEG_08
# ════════════════════════════════════════════════════════════

MATRIX_FN = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix}
records = meta.to_dict("records")

for method in METHODS:
    for k in K_VALUES:
        suffix   = f"_thr{EDGE_THRESHOLD:.2f}".replace(".","") if EDGE_THRESHOLD > 0 else ""
        fname    = f"dataset_{method}_k{k}{suffix}.pt"
        out_path = GRAPHS_DIR / fname

        if out_path.exists() and not FORCE_REBUILD:
            print(f"  Già esistente, skip: {fname}")
            continue

        print(f"\n── {method.upper()} k={k} threshold={EDGE_THRESHOLD} ({N_CHANS} canali) ──")
        paths_map = defaultdict(list)
        for i, r in enumerate(records):
            paths_map[r["path_h5"]].append((i, int(r["epoch_idx"])))

        tmp = {}   # dict: evita pre-allocazione e OOM tra iterazioni
        t0  = time.time()
        for path, items in tqdm(paths_map.items(), desc=f"{method} k={k}", leave=False):
            with h5py.File(path, "r") as f:
                for idx, e_idx in items:
                    x_np = f["data"][e_idx][keep_idx, :].astype(np.float32)
                    mat  = MATRIX_FN[method](x_np)
                    ei   = knn_edge_index(mat, k, threshold=EDGE_THRESHOLD)
                    try:
                        subj = int(records[idx]["subject_id"])
                    except (ValueError, KeyError):
                        continue
                    tmp[idx] = Data(
                        x          = torch.tensor(x_np),
                        edge_index = ei,
                        y          = torch.tensor(int(records[idx]["label_idx"]), dtype=torch.long),
                        subj       = torch.tensor(subj, dtype=torch.long),
                    )

        dataset = [tmp[i] for i in range(len(records)) if i in tmp]
        del tmp
        gc.collect()

        torch.save(dataset, out_path)
        size_mb = out_path.stat().st_size / 1e6
        avg_edges = np.mean([d.edge_index.shape[1] for d in dataset])
        print(f"  ✓ {len(dataset)} trial | {avg_edges:.0f} archi medi | {size_mb:.0f} MB | {int(time.time()-t0)}s")
        print(f"    Salvato: {out_path}")
        del dataset
        gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════
# BUILD IPERGRAFI — PCC hyperedge_index per-trial
# ════════════════════════════════════════════════════════════

if not BUILD_HGNN:
    print("BUILD_HGNN=False — skip")
else:
    for k in K_VALUES:
        suffix   = f"_thr{EDGE_THRESHOLD:.2f}".replace(".","") if EDGE_THRESHOLD > 0 else ""
        fname    = f"dataset_hgnn_k{k}{suffix}.pt"
        out_path = GRAPHS_DIR / fname

        if out_path.exists() and not FORCE_REBUILD:
            print(f"  Già esistente, skip: {fname}")
            continue

        print(f"\n── HGNN k={k} threshold={EDGE_THRESHOLD} ({N_CHANS} canali) ──")
        N_HYPER   = N_CHANS
        paths_map = defaultdict(list)
        for i, r in enumerate(records):
            paths_map[r["path_h5"]].append((i, int(r["epoch_idx"])))

        tmp = [None] * len(records)
        t0  = time.time()
        for path, items in tqdm(paths_map.items(), desc=f"HGNN k={k}", leave=False):
            with h5py.File(path, "r") as f:
                for idx, e_idx in items:
                    x_np = f["data"][e_idx][keep_idx, :].astype(np.float32)
                    he   = hyperedge_index(x_np, k, threshold=EDGE_THRESHOLD)
                    try:
                        subj = int(records[idx]["subject_id"])
                    except (ValueError, KeyError):
                        continue
                    tmp[idx] = Data(
                        x               = torch.tensor(x_np),
                        hyperedge_index = he,
                        num_hyperedges  = torch.tensor(N_HYPER, dtype=torch.long),
                        y               = torch.tensor(int(records[idx]["label_idx"]), dtype=torch.long),
                        subj            = torch.tensor(subj, dtype=torch.long),
                    )

        dataset = [d for d in tmp if d is not None]
        torch.save(dataset, out_path)
        size_mb = out_path.stat().st_size / 1e6
        avg_he  = np.mean([d.hyperedge_index.shape[1] for d in dataset])
        print(f"  ✓ {len(dataset)} trial | {avg_he:.0f} entries he medi | {size_mb:.0f} MB | {int(time.time()-t0)}s")
        print(f"    Salvato: {out_path}")


In [ ]:
# ════════════════════════════════════════════════════════════
# SANITY CHECK — verifica file salvati
# ════════════════════════════════════════════════════════════

print("=== File in data/interim/graphs/ ===\n")
for f in sorted(GRAPHS_DIR.glob("*.pt")):
    ds = torch.load(f, map_location="cpu", weights_only=False)
    d0 = ds[0]
    size_mb = f.stat().st_size / 1e6
    if hasattr(d0, "hyperedge_index"):
        print(f"  {f.name:<35}  {len(ds):>6} trial  "
              f"x={tuple(d0.x.shape)}  he={tuple(d0.hyperedge_index.shape)}  {size_mb:.0f}MB")
    else:
        print(f"  {f.name:<35}  {len(ds):>6} trial  "
              f"x={tuple(d0.x.shape)}  ei={tuple(d0.edge_index.shape)}  {size_mb:.0f}MB")
